In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import sys
import os
from datasets import load_dataset

root_path = os.path.abspath(os.path.join(".."))
if root_path not in sys.path:
    sys.path.append(root_path)

from utils import standardize_model_names  # noqa: E402

In [2]:
ec_url = "https://huggingface.co/datasets/mehmetdavut/RubyCraft-3.4-Eval-Logs/resolve/main/ec_before_after.csv"
df_ec = pd.read_csv(ec_url)
df_ec = standardize_model_names(df_ec, model_column="model")
df_ec = df_ec[~df_ec["model"].str.contains("teacher", case=False, na=False)].copy()

In [3]:
ic_url = "https://huggingface.co/datasets/mehmetdavut/RubyCraft-3.4-Eval-Logs/resolve/main/ic_before_after.csv"
df_ic = pd.read_csv(ic_url)
df_ic = standardize_model_names(df_ic, model_column="model")
df_ic = df_ic[~df_ic["model"].str.contains("teacher", case=False, na=False)].copy()

In [4]:
dataset_meta = load_dataset("mehmetdavut/RubyCraft-3.4-Eval-Logs", "intrinsic_capability_humaneval", split="train")
df_meta = dataset_meta.to_pandas()
df_meta = standardize_model_names(df_meta, model_column="model_architecture")
df_meta = df_meta[~df_meta["model_role"].str.contains("Teacher", case=False, na=False)].copy()

df_meta["passed_bool"] = df_meta["raw_avg_score"] > 0
meta_grouped = df_meta.groupby(["experiment_id", "Architecture", "dataset_size", "data_split", "quantization"])["passed_bool"].sum().reset_index()
meta_grouped.rename(columns={"passed_bool": "raw_score"}, inplace=True)
meta_grouped = meta_grouped.sort_values(["experiment_id", "Architecture", "raw_score", "data_split"])
meta_grouped["rank"] = meta_grouped.groupby(["experiment_id", "Architecture"])["raw_score"].rank(method="first")

df_ic = df_ic.sort_values(["experiment", "Architecture", "before_passed", "model"])
df_ic["rank"] = df_ic.groupby(["experiment", "Architecture"])["before_passed"].rank(method="first")

df_mapping = df_ic.merge(
    meta_grouped,
    left_on=["experiment", "Architecture", "rank"],
    right_on=["experiment_id", "Architecture", "rank"],
    how="left"
)[["experiment", "model", "Architecture", "dataset_size", "data_split", "quantization"]]

In [5]:
df_merged = df_ec.merge(
    df_mapping,
    on=["experiment", "model", "Architecture"],
    how="left"
)

In [6]:
base_mask = df_merged["data_split"].isna() | (df_merged["data_split"] == "Base")
df_merged.loc[base_mask, "dataset_size"] = "-"
df_merged.loc[base_mask, "data_split"] = "Vanilla"
df_merged.loc[base_mask, "quantization"] = "-"

In [7]:
df_merged["after_style_mean"] = df_merged["after_style_mean"].fillna(0).astype(float)
df_merged["before_style_mean"] = df_merged["before_style_mean"].fillna(0).astype(float)

tier_col = "Tier_x" if "Tier_x" in df_merged.columns else "Tier"

df_solved = df_merged[[
    "experiment", "Architecture", tier_col, 
    "dataset_size", "data_split", "quantization", 
    "before_style_mean", "after_style_mean"
]].copy()

df_solved.rename(columns={tier_col: "Tier"}, inplace=True)
df_solved = df_solved.sort_values(by="before_style_mean", ascending=False).reset_index(drop=True)

display(df_solved.head(50))

,experiment,Architecture,Tier,dataset_size,data_split,quantization,before_style_mean,after_style_mean
0,exp-112,Qwen-Coder,Big (≥ 7B),5K,ALL,8-bit,8.02,8.60
1,exp-114,Qwen-Coder,Big (≥ 7B),5K,ALL,16-bit,7.95,8.40
2,exp-110,Qwen-Coder,Big (≥ 7B),5K,HQ,4-bit,7.83,8.43
3,exp-108,Qwen-Coder,Small (< 4B),5K,HQ,16-bit,7.80,8.30
4,exp-110,Qwen-Coder,Big (≥ 7B),5K,ALL,4-bit,7.79,8.22
5,exp-114,Qwen-Coder,Big (≥ 7B),5K,HQ,16-bit,7.58,8.25
6,exp-105,Qwen-Coder,Small (< 4B),1K,HQ,8-bit,7.52,7.85
7,exp-103,Qwen-Coder,Small (< 4B),1K,HQ,4-bit,7.51,8.01
8,exp-110,Qwen,Big (≥ 7B),5K,HQ,4-bit,7.48,8.02
9,exp-109,Qwen-Coder,Big (≥ 7B),1K,ALL,4-bit,7.48,8.57


In [8]:
df_big = df_solved[df_solved["Tier"] == "Big (≥ 7B)"].copy()

df_big = df_big.sort_values(by="after_style_mean", ascending=False).reset_index(drop=True)

display(df_big.head(30))

,experiment,Architecture,Tier,dataset_size,data_split,quantization,before_style_mean,after_style_mean
0,exp-112,Qwen-Coder,Big (≥ 7B),5K,ALL,8-bit,8.02,8.60
1,exp-109,Qwen-Coder,Big (≥ 7B),1K,ALL,4-bit,7.48,8.57
2,exp-110,Qwen-Coder,Big (≥ 7B),5K,HQ,4-bit,7.83,8.43
3,exp-114,Qwen-Coder,Big (≥ 7B),5K,ALL,16-bit,7.95,8.40
4,exp-113,Qwen-Coder,Big (≥ 7B),1K,ALL,16-bit,7.03,8.32
5,exp-111,Qwen-Coder,Big (≥ 7B),1K,HQ,8-bit,7.42,8.30
6,exp-111,Qwen-Coder,Big (≥ 7B),1K,ALL,8-bit,6.88,8.27
7,exp-114,Qwen-Coder,Big (≥ 7B),5K,HQ,16-bit,7.58,8.25
8,exp-110,Qwen-Coder,Big (≥ 7B),5K,ALL,4-bit,7.79,8.22
9,exp-114,Qwen-Coder,Big (≥ 7B),-,Vanilla,-,4.53,8.21


In [9]:
df_small = df_solved[df_solved["Tier"] == "Small (< 4B)"].copy()
df_small = df_small.sort_values(by="after_style_mean", ascending=False).reset_index(drop=True)

display(df_small.head(30))

,experiment,Architecture,Tier,dataset_size,data_split,quantization,before_style_mean,after_style_mean
0,exp-108,Qwen-Coder,Small (< 4B),5K,HQ,16-bit,7.80,8.30
1,exp-103,Qwen-Coder,Small (< 4B),1K,HQ,4-bit,7.51,8.01
2,exp-106,Qwen-Coder,Small (< 4B),5K,HQ,8-bit,7.30,7.87
3,exp-105,Qwen-Coder,Small (< 4B),1K,HQ,8-bit,7.52,7.85
4,exp-107,Qwen-Coder,Small (< 4B),1K,HQ,16-bit,7.45,7.78
5,exp-104,Qwen-Coder,Small (< 4B),5K,ALL,4-bit,6.94,7.71
6,exp-104,Phi,Small (< 4B),5K,ALL,4-bit,6.81,7.64
7,exp-104,Qwen-Coder,Small (< 4B),5K,HQ,4-bit,7.20,7.62
8,exp-103,Qwen-Coder,Small (< 4B),1K,ALL,4-bit,7.19,7.62
9,exp-106,Qwen-Coder,Small (< 4B),5K,ALL,8-bit,7.13,7.60
